In [61]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import csv
import geopandas as gpd
import pylab 
import scipy.stats as stats
from scipy import ndimage
import gmaps 
import gmaps.datasets 
from datetime import datetime
from shapely.geometry import Point
import re

%matplotlib inline

In [62]:
# Set the Filepath
points_file_path = "data/project/Israel_data_gis.csv"
 
FILE_NAME = "data/project/Israel_data_gis.csv"
FILE_HEADER = ['project', 'grower', 'plot_name', 'report_date', 'crop', 'pest', 'data', 
               'latitude', 'longitude', 'plot_latitude', 'plot_longitude']
 
df = pd.read_csv(FILE_NAME)

In [63]:
df.head()

,project,grower,plot_name,report_date,crop,pest,data,latitude,longitude,plot_latitude,plot_longitude
0,Sela,Baumel Avraham,16877,12/1/18,Easy peelers,Medfly infection,0.00,32.655384,34.996071,32.656105,34.995663
1,Sela,Baumel Avraham,16877,22/12/2017,Easy peelers,Medfly Trap Male,0.71,32.655373,34.996095,32.656105,34.995663
2,Sela,Baumel Avraham,16877,12/1/18,Easy peelers,Medfly Trap Male,0.48,32.655386,34.996069,32.656105,34.995663
3,Sela,Baumel Avraham,16877,22/12/2017,Easy peelers,Medfly infection,0.00,32.655371,34.996079,32.656105,34.995663
4,Sela,Baumel Moti,1416,28/12/2017,Nactarine in nethouse,Medfly Trap Male,0.00,32.517142,35.006362,32.529256,35.017313


In [69]:
df['plot_name']

0                   16877
1                   16877
2                   16877
3                   16877
4                    1416
5                    5443
6                 22543-3
7                   90110
8                    1578
9                    5314
10                  16783
11                   5441
12                  90025
13                  90025
14                  18502
15                   5364
16                    204
17                  11137
18                   7322
19                14947-1
20                  16927
21                  16927
22                  16927
23                  16927
24                  16927
25                  16927
26                  12614
27                  12631
28                  15200
29                  12849
               ...       
49185    ??? ?????????_44
49186    ??? ?????????_70
49187       ???? ?????_64
49188    ??? ?????????_75
49189      Neve Mivtah_34
49190      Neve Mivtah_70
49191      Neve Mivtah_34
49192    ???

# Coordinate Cleaning

In [4]:
null_index = df[df['longitude'].isnull()].index
nulls = df.loc[null_index, :]
nulls.head()

,project,grower,plot_name,report_date,crop,pest,data,latitude,longitude,plot_latitude,plot_longitude
9607,Tali Grapes,unknown-???????,???? 223,29/05/2016,Autumn Crisp,Medfly Trap Female,0.0,NaN,NaN,31.548371,34.859504
9608,Tali Grapes,unknown-???????,???? 223,4/9/17,Autumn Crisp,Medfly Trap Female,0.0,NaN,NaN,31.548371,34.859504
9609,Tali Grapes,unknown-???????,???? 223,17/06/2018,Autumn Crisp,Medfly Trap Female,0.0,NaN,NaN,31.548371,34.859504
9610,Tali Grapes,unknown-???????,???? 253,12/9/16,Autumn Crisp,Medfly Trap Female,0.0,NaN,NaN,31.571279,34.849430
9611,Tali Grapes,unknown-???????,???? 223,11/4/16,Autumn Crisp,Medfly Trap Female,0.0,NaN,NaN,31.548371,34.859504


In [5]:
df['longitude'] = df['longitude'].fillna(0)
df['latitude'] = df['latitude'].fillna(0)

In [6]:
#Setting null point coordinates to the centroid of the respective plots that contains them.
for index, row in df.iterrows():
    if row['longitude'] == 0:
        df.loc[index, 'longitude'] = row['plot_longitude']
        df.loc[index, 'latitude'] = row['plot_latitude']

In [7]:
df.loc[null_index, :].head()

,project,grower,plot_name,report_date,crop,pest,data,latitude,longitude,plot_latitude,plot_longitude
9607,Tali Grapes,unknown-???????,???? 223,29/05/2016,Autumn Crisp,Medfly Trap Female,0.0,31.548371,34.859504,31.548371,34.859504
9608,Tali Grapes,unknown-???????,???? 223,4/9/17,Autumn Crisp,Medfly Trap Female,0.0,31.548371,34.859504,31.548371,34.859504
9609,Tali Grapes,unknown-???????,???? 223,17/06/2018,Autumn Crisp,Medfly Trap Female,0.0,31.548371,34.859504,31.548371,34.859504
9610,Tali Grapes,unknown-???????,???? 253,12/9/16,Autumn Crisp,Medfly Trap Female,0.0,31.571279,34.849430,31.571279,34.849430
9611,Tali Grapes,unknown-???????,???? 223,11/4/16,Autumn Crisp,Medfly Trap Female,0.0,31.548371,34.859504,31.548371,34.859504


# Date Cleaning

In [8]:
df.report_date.map(len).value_counts()

10    30525
6     11149
7      6643
8       898
Name: report_date, dtype: int64

In [9]:
print("All instances of date formats in the DataFrame:")
print("Length 6: " + df[(df['report_date'].map(len)) == 6].iloc[0]['report_date'])
print("Length 7: " + df[(df['report_date'].map(len)) == 7].iloc[0]['report_date'])
print("Length 8: " + df[(df['report_date'].map(len)) == 8].iloc[0]['report_date'])
print("Length 10: " + df[(df['report_date'].map(len)) == 10].iloc[0]['report_date'])

All instances of date formats in the DataFrame:
Length 6: 9/1/18
Length 7: 12/1/18
Length 8: 11/11/18
Length 10: 22/12/2017


In [10]:
def convert_points(date):
    if date is None:
        return
    if len(date) == 10:
        return datetime.strptime(date, '%d/%m/%Y')
    else:
        return datetime.strptime(date, '%d/%m/%y')

In [11]:
df['report_date'] = df['report_date'].map(convert_points)

In [12]:
df.head()

,project,grower,plot_name,report_date,crop,pest,data,latitude,longitude,plot_latitude,plot_longitude
0,Sela,Baumel Avraham,16877,2018-01-12,Easy peelers,Medfly infection,0.00,32.655384,34.996071,32.656105,34.995663
1,Sela,Baumel Avraham,16877,2017-12-22,Easy peelers,Medfly Trap Male,0.71,32.655373,34.996095,32.656105,34.995663
2,Sela,Baumel Avraham,16877,2018-01-12,Easy peelers,Medfly Trap Male,0.48,32.655386,34.996069,32.656105,34.995663
3,Sela,Baumel Avraham,16877,2017-12-22,Easy peelers,Medfly infection,0.00,32.655371,34.996079,32.656105,34.995663
4,Sela,Baumel Moti,1416,2017-12-28,Nactarine in nethouse,Medfly Trap Male,0.00,32.517142,35.006362,32.529256,35.017313


# Plot Name Cleaning

In [70]:
df['plot_name']




0                   16877
1                   16877
2                   16877
3                   16877
4                    1416
5                    5443
6                 22543-3
7                   90110
8                    1578
9                    5314
10                  16783
11                   5441
12                  90025
13                  90025
14                  18502
15                   5364
16                    204
17                  11137
18                   7322
19                14947-1
20                  16927
21                  16927
22                  16927
23                  16927
24                  16927
25                  16927
26                  12614
27                  12631
28                  15200
29                  12849
               ...       
49185    ??? ?????????_44
49186    ??? ?????????_70
49187       ???? ?????_64
49188    ??? ?????????_75
49189      Neve Mivtah_34
49190      Neve Mivtah_70
49191      Neve Mivtah_34
49192    ???

In [71]:
missing_plots = df[df['plot_name'].map(lambda st: len(st) == 0)].index
df = df.drop(index=missing_plots)

In [72]:
df['plot_name'] = df['plot_name'].str.replace('-','.').str.replace('[^\.\d]', '').astype(dtype=float, errors='ignore')

In [73]:
df.head()

,project,grower,plot_name,report_date,crop,pest,data,latitude,longitude,plot_latitude,plot_longitude
0,Sela,Baumel Avraham,16877,12/1/18,Easy peelers,Medfly infection,0.00,32.655384,34.996071,32.656105,34.995663
1,Sela,Baumel Avraham,16877,22/12/2017,Easy peelers,Medfly Trap Male,0.71,32.655373,34.996095,32.656105,34.995663
2,Sela,Baumel Avraham,16877,12/1/18,Easy peelers,Medfly Trap Male,0.48,32.655386,34.996069,32.656105,34.995663
3,Sela,Baumel Avraham,16877,22/12/2017,Easy peelers,Medfly infection,0.00,32.655371,34.996079,32.656105,34.995663
4,Sela,Baumel Moti,1416,28/12/2017,Nactarine in nethouse,Medfly Trap Male,0.00,32.517142,35.006362,32.529256,35.017313


# GeoDataFrame

In [74]:
#Convert cleaned DateFrame to a GeoDataFrame
gdf = gpd.GeoDataFrame(
    df, geometry=gpd.points_from_xy(df.longitude, df.latitude))

In [75]:
gdf.head()

,project,grower,plot_name,report_date,crop,pest,data,latitude,longitude,plot_latitude,plot_longitude,geometry
0,Sela,Baumel Avraham,16877,12/1/18,Easy peelers,Medfly infection,0.00,32.655384,34.996071,32.656105,34.995663,POINT (34.996071 32.6553838)
1,Sela,Baumel Avraham,16877,22/12/2017,Easy peelers,Medfly Trap Male,0.71,32.655373,34.996095,32.656105,34.995663,POINT (34.99609539999999 32.6553732)
2,Sela,Baumel Avraham,16877,12/1/18,Easy peelers,Medfly Trap Male,0.48,32.655386,34.996069,32.656105,34.995663,POINT (34.9960686 32.6553858)
3,Sela,Baumel Avraham,16877,22/12/2017,Easy peelers,Medfly infection,0.00,32.655371,34.996079,32.656105,34.995663,POINT (34.9960794 32.6553708)
4,Sela,Baumel Moti,1416,28/12/2017,Nactarine in nethouse,Medfly Trap Male,0.00,32.517142,35.006362,32.529256,35.017313,POINT (35.0063623 32.5171422)


# Project Datasets

In [27]:
gdf["project"].value_counts()

Tali Grapes    39139
Zifit Darom     9943
Sela              63
Yaham             55
Zemach            15
Name: project, dtype: int64

In [28]:
tali = gdf[gdf['project'] == "Tali Grapes"]
zifit = gdf[gdf['project'] == "Zifit Darom"]
sela = gdf[gdf['project'] == "Sela"]
yaham = gdf[gdf['project'] == "Yaham"]
zemach = gdf[gdf['project'] == "Zemach"]

# Time Datasets

In [29]:
gdf['year'] = gdf['report_date'].dt.year
gdf['month'] = gdf['report_date'].dt.month
gdf['day'] = gdf['report_date'].dt.day

In [48]:
gdf.year.sort_values().unique()

array([2014, 2015, 2016, 2017, 2018, 2019])

In [49]:
data_2014 = gdf[gdf['year'] == 2014]
data_2015 = gdf[gdf['year'] == 2015]
data_2016 = gdf[gdf['year'] == 2016]
data_2017 = gdf[gdf['year'] == 2017]
data_2018 = gdf[gdf['year'] == 2018]
data_2019 = gdf[gdf['year'] == 2019]